# Módulo 2: Clasificación CNN con Transfer Learning (ResNet18)
## Detección de Conducción Distractiva
### IRNA - Universidad Nacional de Colombia

Este notebook implementa un clasificador de 10 clases para detectar tipos de distracciones
al conducir, utilizando ResNet18 preentrenado en ImageNet con fine-tuning supervisado.

**Pipeline:**
1. Carga/generación de datos
2. Preprocesamiento y DataLoaders
3. Arquitectura ResNet18 con cabeza personalizada
4. Entrenamiento con métricas por época
5. Evaluación completa (F1, precisión, recall, matriz de confusión)
6. Visualización de ejemplos correctos e incorrectos
7. Análisis de distracciones y medidas preventivas
8. Guardado de artefactos

## 1. Imports y Configuración del Entorno

In [ ]:
# Imports principales de PyTorch y utilidades
import os
import sys
import time
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from collections import Counter
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.transforms as transforms
import torchvision.models as models

# Scikit-learn para métricas
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, precision_score, recall_score, accuracy_score)

# Configuración de dispositivo — usar CPU (compatible con todos los entornos)
device = torch.device('cpu')

# Directorios de salida
OUTPUT_DIR = '.'
MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Semilla aleatoria para reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'PyTorch versión: {torch.__version__}')
print(f'Dispositivo    : {device}')
print(f'Directorio modelos: {os.path.abspath(MODELS_DIR)}')

## 2. Definición de Clases

In [ ]:
# Nombres de las 5 clases reales del dataset
CLASS_NAMES = [
    'c0: Conducción segura',
    'c1: Girando / Mirando espejos',
    'c2: Texteando al conducir',
    'c3: Hablando por teléfono',
    'c4: Otras actividades'
]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_SHORT = [f'c{i}' for i in range(NUM_CLASSES)]

# Medidas preventivas por tipo de comportamiento (para el análisis final)
PREVENTIVE_MEASURES = {
    0: 'Mantener hábitos seguros; refuerzo positivo al conductor.',
    1: 'Monitorear el uso correcto de direccionales y espejos; asegurar atención al frente.',
    2: 'Activar modo «No molestar» en el teléfono y usar mensajes de voz manos libres.',
    3: 'Usar auriculares Bluetooth o altavoz del vehículo; prohibir teléfono en mano.',
    4: 'Evitar beber, maquillarse, o manipular el radio mientras el vehículo esté en movimiento.'
}

print(f'Número de clases: {NUM_CLASSES}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  [{i}] {name}')

## 3. Generador de Imágenes Sintéticas

In [ ]:
def generate_synthetic_driver_images(n_per_class=200, img_size=224):
    """
    Genera imágenes sintéticas de conductores con patrones visuales por clase.
    Cada clase tiene fondo de color distinto + formas geométricas que simulan la actividad.
    
    Parámetros
    ----------
    n_per_class : int  — imágenes a generar por clase
    img_size    : int  — tamaño cuadrado de las imágenes
    
    Retorna
    -------
    images : list[np.ndarray]  — arrays uint8 (H, W, 3)
    labels : list[int]         — etiquetas [0–9]
    """
    # Color de fondo característico por clase (RGB)
    class_colors = {
        0: (70,  130, 50),   # verde       — conducción segura
        1: (200, 100, 80),   # naranja-rojo — texto mano derecha
        2: (180, 80,  180),  # morado       — teléfono mano derecha
        3: (80,  80,  200),  # azul         — texto mano izquierda
        4: (200, 180, 60),   # amarillo     — teléfono mano izquierda
        5: (100, 200, 200),  # cyan         — radio/controles
        6: (200, 150, 100),  # marrón       — bebiendo
        7: (150, 100, 200),  # violeta      — alcanzando
        8: (255, 150, 200),  # rosa         — cabello/maquillaje
        9: (100, 200, 150),  # verde-azul   — conversando
    }
    # Número de objetos visuales por clase (mayor = más actividad visual)
    class_shapes = {0: 5, 1: 16, 2: 13, 3: 16, 4: 13, 5: 9, 6: 11, 7: 8, 8: 19, 9: 7}
    
    images, labels = [], []
    rng = np.random.RandomState(SEED)
    
    for cls in range(10):
        base = class_colors[cls]
        ns   = class_shapes[cls]
        for _ in range(n_per_class):
            # Variación de color de fondo para diversidad intra-clase
            var = rng.randint(-25, 25, 3)
            bg  = tuple(np.clip(np.array(base) + var, 0, 255).astype(int).tolist())
            img  = Image.new('RGB', (img_size, img_size), bg)
            draw = ImageDraw.Draw(img)
            # Formas geométricas aleatorias
            for _ in range(ns + rng.randint(0, 12)):
                x  = rng.randint(0, img_size)
                y  = rng.randint(0, img_size)
                r  = rng.randint(5, 45)
                c  = tuple(rng.randint(50, 255, 3).tolist())
                st = rng.randint(0, 3)
                if st == 0:
                    draw.ellipse([x-r, y-r, x+r, y+r], fill=c)
                elif st == 1:
                    draw.rectangle([x-r, y-r, x+r, y+r], fill=c)
                else:
                    draw.line([x-r, y-r, x+r, y+r], fill=c, width=3)
            # Ruido gaussiano para simular variaciones de iluminación
            arr  = np.array(img).astype(np.float32)
            arr += rng.normal(0, 15, arr.shape)
            arr  = np.clip(arr, 0, 255).astype(np.uint8)
            images.append(arr)
            labels.append(cls)
    
    return images, labels

print('Generador de imágenes sintéticas definido.')

## 4. Carga / Generación de Datos

In [ ]:
# Carga del dataset real de Kaggle
raw_images = []
raw_labels = []
data_source = 'Kaggle (Real)'

import kagglehub
print('Cargando dataset de Kaggle real...')
path = r'C:\Users\santy\.cache\kagglehub\datasets\arafatsahinafridi\multi-class-driver-behavior-image-dataset\versions\1\Multi-Class Driver Behavior Image Dataset'
CLASSES = ['safe_driving', 'turning', 'texting_phone', 'talking_phone', 'other_activities']

for cls_idx, cls_name in enumerate(CLASSES):
    folder = os.path.join(path, cls_name)
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:400]
        for fname in files:
            img = Image.open(os.path.join(folder, fname)).convert('RGB').resize((224, 224))
            raw_images.append(np.array(img))
            raw_labels.append(cls_idx)

print(f'Dataset Kaggle real cargado: {len(raw_images)} imágenes')
print(f'\nFuente     : {data_source}')
print(f'Total imgs : {len(raw_images)}')
from collections import Counter
print(f'Distribución: {Counter(raw_labels)}')

## 5. Dataset Personalizado y Preprocesamiento

In [ ]:
# Dataset personalizado para PyTorch
class DriverDataset(Dataset):
    """
    Dataset de conducción distractiva compatible con PyTorch DataLoader.
    Acepta una lista de arrays numpy (H, W, 3) y aplica un transform.
    """
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        img   = Image.fromarray(self.images[idx].astype(np.uint8))
        label = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, label

print('Clase DriverDataset definida.')

In [ ]:
# Normalización ImageNet (necesaria para pesos preentrenados)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Transforms de entrenamiento con data augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Transforms de validación/test (sin augmentation)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print('Transforms definidos:')
print('  Train: Resize + HFlip + Rotation + ColorJitter + Affine + Normalize')
print('  Val  : Resize + Normalize')

In [ ]:
# Split estratificado 80% train / 20% val
all_indices = np.arange(len(raw_images))
all_labels  = np.array(raw_labels)

train_idx, val_idx = train_test_split(
    all_indices, test_size=0.20, random_state=SEED, stratify=all_labels
)

# Crear datasets
train_images = [raw_images[i] for i in train_idx]
train_labels = [raw_labels[i] for i in train_idx]
val_images   = [raw_images[i] for i in val_idx]
val_labels   = [raw_labels[i] for i in val_idx]

train_dataset = DriverDataset(train_images, train_labels, transform=train_transform)
val_dataset   = DriverDataset(val_images,   val_labels,   transform=val_transform)

# DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print(f'Split estratificado 80/20:')
print(f'  Train: {len(train_dataset)} imágenes ({len(train_loader)} batches de {BATCH_SIZE})')
print(f'  Val  : {len(val_dataset)}   imágenes ({len(val_loader)} batches de {BATCH_SIZE})')
print(f'\nDistribución en train: {Counter(train_labels)}')
print(f'Distribución en val  : {Counter(val_labels)}')

## 6. Arquitectura ResNet18 con Transfer Learning

In [ ]:
def build_model(num_classes=10):
    """
    Construye ResNet18 con Transfer Learning.
    - Intenta cargar pesos preentrenados en ImageNet.
    - Si no hay conectividad, usa pesos aleatorios.
    - Congela todas las capas excepto layer4 y la cabeza de clasificación.
    - Reemplaza el clasificador final con una cabeza personalizada de 10 clases.
    
    Parámetros
    ----------
    num_classes : int — número de clases de salida
    
    Retorna
    -------
    model : nn.Module — ResNet18 con cabeza personalizada
    """
    # Intentar cargar pesos preentrenados
    try:
        from torchvision.models import ResNet18_Weights
        model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        print('Pesos preentrenados ImageNet cargados correctamente.')
    except Exception:
        try:
            model = models.resnet18(pretrained=True)
            print('Pesos preentrenados cargados (API legacy).')
        except Exception as e2:
            model = models.resnet18(pretrained=False)
            print(f'Sin pesos preentrenados (pesos aleatorios). Razón: {e2}')
    
    # Congelar todas las capas excepto layer4 y fc
    for name, param in model.named_parameters():
        if 'layer4' not in name and 'fc' not in name:
            param.requires_grad = False
    
    # Cabeza de clasificación personalizada
    in_features = model.fc.in_features  # 512 en ResNet18
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(256),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )
    
    return model

# Construir y mover al dispositivo
model = build_model(num_classes=NUM_CLASSES)
model = model.to(device)

# Resumen de parámetros
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nResumen del modelo:')
print(f'  Parámetros totales    : {total_params:,}')
print(f'  Parámetros entrenables: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)')
print(f'  Capas congeladas      : {(total_params-trainable_params):,} parámetros')

## 7. Configuración del Entrenamiento

In [ ]:
# Solo optimizar parámetros entrenables (layer4 + fc)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)

# Scheduler: reducir LR cuando val_loss no mejore en 3 épocas
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)

NUM_EPOCHS   = 4
BEST_VAL_ACC = 0.0
BEST_MODEL_PATH = os.path.join(MODELS_DIR, 'resnet18_driver.pt')

print(f'Configuración de entrenamiento:')
print(f'  Épocas        : {NUM_EPOCHS}')
print(f'  Batch size    : {BATCH_SIZE}')
print(f'  Learning rate : 1e-4 (con ReduceLROnPlateau)')
print(f'  Pérdida       : CrossEntropyLoss')
print(f'  Optimizador   : Adam (weight_decay=1e-4)')
print(f'  Mejor modelo  : {BEST_MODEL_PATH}')

## 8. Funciones de Entrenamiento y Evaluación

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Entrena el modelo por una época y retorna loss y accuracy medios."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss     = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == lbls).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """Evalúa el modelo en un DataLoader y retorna loss, accuracy, predicciones y etiquetas."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs     = model(imgs)
            loss        = criterion(outputs, lbls)
            total_loss += loss.item() * imgs.size(0)
            preds       = outputs.argmax(dim=1)
            correct    += (preds == lbls).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(lbls.cpu().numpy().tolist())
    return total_loss / total, correct / total, all_preds, all_labels

print('Funciones train_one_epoch y evaluate definidas.')

## 9. Bucle de Entrenamiento Principal

In [ ]:
# Historial de métricas por época
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss'  : [], 'val_acc'  : []
}

print(f'Iniciando entrenamiento en {device} por {NUM_EPOCHS} épocas...')
print('=' * 75)
print(f'{'Época':>6} | {'T-Loss':>8} | {'T-Acc%':>7} | {'V-Loss':>8} | {'V-Acc%':>7} | {'LR':>9} | Tiempo')
print('-' * 75)

t_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    t_ep = time.time()
    
    # Entrenamiento
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validación
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    
    # Scheduler paso
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Guardar historial
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Guardar mejor modelo
    marker = ''
    if val_acc > BEST_VAL_ACC:
        BEST_VAL_ACC = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        marker = ' <<< MEJOR'
    
    elapsed = time.time() - t_ep
    print(f'{epoch:>6} | {train_loss:>8.4f} | {train_acc*100:>7.2f} | '
          f'{val_loss:>8.4f} | {val_acc*100:>7.2f} | {current_lr:>9.2e} | '
          f'{elapsed:.1f}s{marker}')

total_time = time.time() - t_start
print('=' * 75)
print(f'Entrenamiento completado en {total_time/60:.1f} min')
print(f'Mejor val accuracy: {BEST_VAL_ACC*100:.2f}%  →  {BEST_MODEL_PATH}')

## 10. Curvas de Entrenamiento

In [ ]:
# Visualización de las curvas de entrenamiento (loss y accuracy)
epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Curva de pérdida ---
axes[0].plot(epochs_range, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-s', markersize=4, label='Val Loss')
axes[0].set_xlabel('Época', fontsize=12)
axes[0].set_ylabel('Pérdida (CrossEntropy)', fontsize=12)
axes[0].set_title('Curvas de Pérdida', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.4)

# --- Curva de accuracy ---
axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', markersize=4, label='Train Acc')
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   'r-s', markersize=4, label='Val Acc')
best_epoch = np.argmax(history['val_acc']) + 1
axes[1].axvline(x=best_epoch, color='green', linestyle='--', linewidth=1.5,
               label=f'Mejor época: {best_epoch} ({BEST_VAL_ACC*100:.1f}%)')
axes[1].set_xlabel('Época', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Curvas de Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_training_curves.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_training_curves.png')

## 11. Evaluación Final con el Mejor Modelo

In [ ]:
# Cargar el mejor modelo guardado para evaluación final
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
print(f'Mejor modelo cargado desde: {BEST_MODEL_PATH}')

# Evaluación completa en el conjunto de validación
_, final_acc, final_preds, final_labels = evaluate(model, val_loader, criterion, device)

# Métricas globales
f1_macro   = f1_score(final_labels, final_preds, average='macro',    zero_division=0)
f1_weighted= f1_score(final_labels, final_preds, average='weighted', zero_division=0)
prec_macro = precision_score(final_labels, final_preds, average='macro',    zero_division=0)
rec_macro  = recall_score(final_labels,   final_preds, average='macro',    zero_division=0)

print()
print('='*55)
print('MÉTRICAS GLOBALES EN CONJUNTO DE VALIDACIÓN')
print('='*55)
print(f'  Accuracy global    : {final_acc*100:.2f}%')
print(f'  F1-score macro     : {f1_macro:.4f}')
print(f'  F1-score weighted  : {f1_weighted:.4f}')
print(f'  Precisión macro    : {prec_macro:.4f}')
print(f'  Recall macro       : {rec_macro:.4f}')
print('='*55)

In [ ]:
# Reporte detallado por clase (scikit-learn)
print('CLASSIFICATION REPORT POR CLASE:')
print('='*70)
report = classification_report(
    final_labels, final_preds,
    target_names=CLASS_SHORT,
    zero_division=0
)
print(report)

## 12. Matriz de Confusión

In [ ]:
# Calcular y visualizar la matriz de confusión
cm = confusion_matrix(final_labels, final_preds)

# Normalizar por fila (recall por clase)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Matriz absoluta
sns.heatmap(cm, ax=axes[0], annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            cbar_kws={'label': 'Conteo'})
axes[0].set_xlabel('Predicción', fontsize=12)
axes[0].set_ylabel('Real', fontsize=12)
axes[0].set_title('Matriz de Confusión\n(valores absolutos)', fontsize=13, fontweight='bold')

# Matriz normalizada (recall)
sns.heatmap(cm_norm, ax=axes[1], annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            vmin=0, vmax=1, cbar_kws={'label': 'Recall por clase'})
axes[1].set_xlabel('Predicción', fontsize=12)
axes[1].set_ylabel('Real', fontsize=12)
axes[1].set_title('Matriz de Confusión Normalizada\n(recall por clase)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_confusion_matrix.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_confusion_matrix.png')

## 13. Métricas por Clase

In [ ]:
# Calcular métricas por clase y visualizarlas
prec_per_class = precision_score(final_labels, final_preds, average=None, zero_division=0)
rec_per_class  = recall_score(final_labels,   final_preds, average=None, zero_division=0)
f1_per_class   = f1_score(final_labels,       final_preds, average=None, zero_division=0)
acc_per_class  = cm.diagonal() / cm.sum(axis=1)

# Tabla resumen
print('MÉTRICAS POR CLASE:')
print(f'{'Clase':<35} {'Acc%':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}')
print('-' * 60)
class_metrics = {}
for i in range(NUM_CLASSES):
    print(f'{CLASS_NAMES[i]:<35} {acc_per_class[i]*100:>6.1f} {prec_per_class[i]:>6.3f} '
          f'{rec_per_class[i]:>6.3f} {f1_per_class[i]:>6.3f}')
    class_metrics[CLASS_NAMES[i]] = {
        'accuracy': float(acc_per_class[i]),
        'precision': float(prec_per_class[i]),
        'recall': float(rec_per_class[i]),
        'f1': float(f1_per_class[i])
    }
print('-' * 60)

In [ ]:
# Gráfico de barras agrupadas: Precisión, Recall y F1 por clase
x     = np.arange(NUM_CLASSES)
width = 0.26

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - width, prec_per_class, width, label='Precisión', color='steelblue',  alpha=0.85, edgecolor='black', linewidth=0.5)
ax.bar(x,         rec_per_class,  width, label='Recall',    color='darkorange', alpha=0.85, edgecolor='black', linewidth=0.5)
ax.bar(x + width, f1_per_class,   width, label='F1-score',  color='seagreen',   alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Clase', fontsize=12)
ax.set_ylabel('Puntuación', fontsize=12)
ax.set_title('Precisión, Recall y F1-score por Clase', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_SHORT)
ax.set_ylim(0, 1.15)
ax.axhline(y=f1_macro, color='red', linestyle='--', linewidth=1.5, label=f'F1 macro: {f1_macro:.3f}')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_metrics_per_class.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_metrics_per_class.png')

## 14. Visualización de Ejemplos Correctos e Incorrectos

In [ ]:
# Recolectar imágenes con sus predicciones del conjunto de validación
all_imgs_np   = []
all_preds_np  = []
all_labels_np = []

# Desnormalización para visualización
mean_t = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_t  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

model.eval()
with torch.no_grad():
    for imgs, lbls in val_loader:
        outputs = model(imgs.to(device))
        preds   = outputs.argmax(dim=1).cpu()
        # Desnormalizar para visualización
        imgs_denorm = imgs * std_t + mean_t
        imgs_denorm = torch.clamp(imgs_denorm, 0, 1)
        # Convertir a uint8 numpy
        for i in range(imgs.size(0)):
            arr = (imgs_denorm[i].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            all_imgs_np.append(arr)
            all_preds_np.append(preds[i].item())
            all_labels_np.append(lbls[i].item())

all_preds_np  = np.array(all_preds_np)
all_labels_np = np.array(all_labels_np)
correct_mask   = all_preds_np == all_labels_np
incorrect_mask = ~correct_mask

print(f'Imágenes evaluadas: {len(all_imgs_np)}')
print(f'  Correctas  : {correct_mask.sum()} ({correct_mask.mean()*100:.1f}%)')
print(f'  Incorrectas: {incorrect_mask.sum()} ({incorrect_mask.mean()*100:.1f}%)')

In [ ]:
def plot_image_grid(imgs_list, preds_list, labels_list, title, filename, n_rows=5, n_cols=4, correct=True):
    """
    Visualiza un grid de imágenes con sus predicciones y etiquetas reales.
    
    Parámetros
    ----------
    imgs_list   : lista de arrays numpy (H, W, 3)
    preds_list  : lista de predicciones
    labels_list : lista de etiquetas reales
    title       : título del grid
    filename    : nombre del archivo de salida
    n_rows, n_cols : dimensiones del grid
    correct     : True para ejemplos correctos, False para incorrectos
    """
    n_show = min(n_rows * n_cols, len(imgs_list))
    # Seleccionar aleatoriamente
    rng     = np.random.RandomState(7)
    indices = rng.choice(len(imgs_list), size=n_show, replace=False)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3.2, n_rows*3.4))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    axes = axes.flatten()
    
    border_color = '#2ecc71' if correct else '#e74c3c'
    
    for ax_i, idx in enumerate(indices):
        ax = axes[ax_i]
        ax.imshow(imgs_list[idx])
        pred  = preds_list[idx]
        real  = labels_list[idx]
        color = '#2ecc71' if pred == real else '#e74c3c'
        ax.set_title(f'Pred: {CLASS_SHORT[pred]}\nReal: {CLASS_SHORT[real]}',
                    fontsize=9, color=color)
        ax.axis('off')
        # Borde de color
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(3)
    
    # Ocultar ejes vacíos
    for ax_i in range(n_show, len(axes)):
        axes[ax_i].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), bbox_inches='tight', dpi=120)
    plt.show()
    print(f'Guardado: {filename}')

print('Función plot_image_grid definida.')

In [ ]:
# Grid de ejemplos CORRECTAMENTE clasificados (5x4 = 20 imágenes)
correct_imgs   = [all_imgs_np[i]   for i in range(len(all_imgs_np)) if correct_mask[i]]
correct_preds  = [all_preds_np[i]  for i in range(len(all_preds_np)) if correct_mask[i]]
correct_labels = [all_labels_np[i] for i in range(len(all_labels_np)) if correct_mask[i]]

if len(correct_imgs) >= 20:
    plot_image_grid(
        correct_imgs, correct_preds, correct_labels,
        title='Ejemplos Clasificados CORRECTAMENTE (verde)',
        filename='fig_correct_examples.png',
        n_rows=5, n_cols=4, correct=True
    )
else:
    print(f'Solo hay {len(correct_imgs)} correctas — mostrando todas.')
    plot_image_grid(
        correct_imgs, correct_preds, correct_labels,
        title='Ejemplos Clasificados CORRECTAMENTE',
        filename='fig_correct_examples.png',
        n_rows=max(1, len(correct_imgs)//4 + 1), n_cols=4, correct=True
    )

In [ ]:
# Grid de ejemplos INCORRECTAMENTE clasificados (5x4 = 20 imágenes)
wrong_imgs   = [all_imgs_np[i]   for i in range(len(all_imgs_np)) if incorrect_mask[i]]
wrong_preds  = [all_preds_np[i]  for i in range(len(all_preds_np)) if incorrect_mask[i]]
wrong_labels = [all_labels_np[i] for i in range(len(all_labels_np)) if incorrect_mask[i]]

if len(wrong_imgs) > 0:
    n_wrong_show = min(20, len(wrong_imgs))
    nr = (n_wrong_show + 3) // 4
    plot_image_grid(
        wrong_imgs[:n_wrong_show], wrong_preds[:n_wrong_show], wrong_labels[:n_wrong_show],
        title='Ejemplos Clasificados INCORRECTAMENTE (rojo)',
        filename='fig_wrong_examples.png',
        n_rows=nr, n_cols=4, correct=False
    )
else:
    print('¡Perfecto! No hay ejemplos incorrectos para mostrar.')
    # Crear figura placeholder
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, 'No hay errores de clasificación', ha='center', va='center', fontsize=16)
    ax.axis('off')
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig_wrong_examples.png'), bbox_inches='tight', dpi=120)
    plt.show()

## 15. Análisis de Distracciones

In [ ]:
# Tabla de accuracy por clase y detección de las más difíciles
print('='*70)
print('ANÁLISIS DE DISTRACCIONES — ACCURACY Y DIFICULTAD DE DETECCIÓN')
print('='*70)

class_acc_sorted = sorted(
    [(i, acc_per_class[i], f1_per_class[i]) for i in range(NUM_CLASSES)],
    key=lambda x: x[1]
)

print(f'\nRanking de clases por accuracy (de menor a mayor):')
print(f'  {'#':>3} {'Clase':<35} {'Acc%':>7} {'F1':>7}')
print('  ' + '-'*55)
for rank, (cls_id, acc, f1_c) in enumerate(class_acc_sorted, 1):
    marker = '  <-- DIFÍCIL' if rank <= 3 else ''
    print(f'  {rank:>3} {CLASS_NAMES[cls_id]:<35} {acc*100:>7.1f} {f1_c:>7.3f}{marker}')

# Las 3 clases más difíciles
hardest_3 = class_acc_sorted[:3]
print(f'\nLas 3 clases MÁS DIFÍCILES de detectar:')
for rank, (cls_id, acc, f1_c) in enumerate(hardest_3, 1):
    print(f'  {rank}. {CLASS_NAMES[cls_id]}')
    print(f'     Accuracy: {acc*100:.1f}%  |  F1: {f1_c:.3f}')
    print(f'     Medida preventiva: {PREVENTIVE_MEASURES[cls_id]}')
    print()

In [ ]:
# Visualización de accuracy por clase con código de colores
sorted_ids  = [x[0] for x in class_acc_sorted]
sorted_accs = [x[1]*100 for x in class_acc_sorted]
sorted_names = [CLASS_SHORT[i] for i in sorted_ids]

bar_colors = ['#e74c3c' if a < 60 else ('#f39c12' if a < 80 else '#2ecc71') for a in sorted_accs]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(sorted_names, sorted_accs, color=bar_colors, edgecolor='black', linewidth=0.7)

for bar, acc in zip(bars, sorted_accs):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
           f'{acc:.1f}%', va='center', fontsize=10)

ax.axvline(x=final_acc*100, color='navy', linestyle='--', linewidth=2,
           label=f'Acc. global: {final_acc*100:.1f}%')
ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy por Clase\n(rojo < 60%, naranja 60–80%, verde > 80%)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 115)
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_accuracy_per_class.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_accuracy_per_class.png')

## 16. Medidas Preventivas por Clase

In [ ]:
# Resumen completo de medidas preventivas por tipo de distracción
print('='*70)
print('MEDIDAS PREVENTIVAS POR TIPO DE DISTRACCIÓN')
print('='*70)
for cls_id in range(NUM_CLASSES):
    print(f'\n[{CLASS_NAMES[cls_id]}]')
    print(f'  Accuracy del modelo: {acc_per_class[cls_id]*100:.1f}%')
    print(f'  Medida preventiva  : {PREVENTIVE_MEASURES[cls_id]}')
print('\n' + '='*70)

## 17. Guardado de Artefactos del Modelo

In [ ]:
# 1. Modelo ya guardado como mejor checkpoint durante entrenamiento
#    Se guarda también el state_dict final para completitud
final_model_path = os.path.join(MODELS_DIR, 'resnet18_driver_final.pt')
torch.save(model.state_dict(), final_model_path)
print(f'Estado final del modelo guardado: {final_model_path}')

# 2. Nombres de clases
class_names_path = os.path.join(MODELS_DIR, 'class_names.pkl')
with open(class_names_path, 'wb') as f:
    pickle.dump(CLASS_NAMES, f)
print(f'Nombres de clases guardados     : {class_names_path}')

# 3. Métricas por clase
class_metrics_path = os.path.join(MODELS_DIR, 'class_metrics.pkl')
with open(class_metrics_path, 'wb') as f:
    pickle.dump(class_metrics, f)
print(f'Métricas por clase guardadas    : {class_metrics_path}')

# 4. Historial de entrenamiento
history_path = os.path.join(MODELS_DIR, 'training_history.pkl')
with open(history_path, 'wb') as f:
    pickle.dump(history, f)
print(f'Historial de entrenamiento      : {history_path}')

print('\nTodos los artefactos guardados correctamente.')

## 18. Resumen Final

In [ ]:
# Inventario completo de artefactos y resumen de resultados
print('='*65)
print('RESUMEN FINAL — MÓDULO 2: CLASIFICACIÓN CNN (ResNet18)')
print('='*65)

print(f'\n1. DATOS')
print(f'   Fuente        : {data_source}')
print(f'   Total imágenes: {len(raw_images)} ({NUM_CLASSES} clases × {len(raw_images)//NUM_CLASSES} imgs)')
print(f'   Train/Val     : {len(train_dataset)}/{len(val_dataset)} (80/20 estratificado)')

print(f'\n2. MODELO')
print(f'   Arquitectura  : ResNet18 con Transfer Learning')
print(f'   Parámetros    : {total_params:,} total / {trainable_params:,} entrenables')
print(f'   Épocas        : {NUM_EPOCHS}')

print(f'\n3. RESULTADOS')
print(f'   Accuracy val  : {final_acc*100:.2f}%')
print(f'   F1 macro      : {f1_macro:.4f}')
print(f'   F1 weighted   : {f1_weighted:.4f}')
print(f'   Precisión mac : {prec_macro:.4f}')
print(f'   Recall macro  : {rec_macro:.4f}')

print(f'\n4. CLASE MÁS DIFÍCIL: {CLASS_NAMES[hardest_3[0][0]]} (acc={hardest_3[0][1]*100:.1f}%)')

print(f'\n5. ARTEFACTOS GENERADOS')
artifacts = [
    os.path.join(MODELS_DIR, 'resnet18_driver.pt'),
    os.path.join(MODELS_DIR, 'resnet18_driver_final.pt'),
    os.path.join(MODELS_DIR, 'class_names.pkl'),
    os.path.join(MODELS_DIR, 'class_metrics.pkl'),
    os.path.join(MODELS_DIR, 'training_history.pkl'),
    os.path.join(OUTPUT_DIR, 'fig_training_curves.png'),
    os.path.join(OUTPUT_DIR, 'fig_confusion_matrix.png'),
    os.path.join(OUTPUT_DIR, 'fig_metrics_per_class.png'),
    os.path.join(OUTPUT_DIR, 'fig_correct_examples.png'),
    os.path.join(OUTPUT_DIR, 'fig_wrong_examples.png'),
    os.path.join(OUTPUT_DIR, 'fig_accuracy_per_class.png'),
]
for art in artifacts:
    ok   = os.path.exists(art)
    size = os.path.getsize(art)/1024 if ok else 0
    mark = 'OK' if ok else 'FALTANTE'
    print(f'   [{mark}] {os.path.basename(art)} ({size:.1f} KB)')

print('='*65)
print('Notebook completado exitosamente.')